## Utilities: stores functions and methods.
`Tomaz Cavalcante`

### Importing libraries

In [1]:
import random
import torch
import numpy as np
import pandas as pd
import pickle
import scipy.sparse as sp
from torch.utils.data import Dataset, DataLoader
import h5py

# >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

SEED       = 42    # semente do experimentador: splits, pesos, etc.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'torch: {torch.__version__}')
print(f'device: {DEVICE}')

torch: 2.11.0+cu128
device: cuda


### Initial Setup

In [2]:
def load_adj_matrix(pkl_filename):
    """Loads the adjacency matrix and applies random walk normalization."""
    with open(pkl_filename, 'rb') as f:
        # Legacy pickle format requires latin1 encoding
        _, _, adj_mx = pickle.load(f, encoding='latin1')
    
    # Random walk normalization: D^-1 A
    # First, calculate row sums (degrees)
    row_sums = np.array(adj_mx.sum(axis=1))
    # Handle division by zero
    d_inv = np.power(row_sums, -1).flatten()
    d_inv[np.isinf(d_inv)] = 0.
    
    # Create diagonal matrix D^-1 and multiply with A
    d_mat_inv = sp.diags(d_inv)
    normalized_adj = d_mat_inv.dot(sp.coo_matrix(adj_mx))
    
    # Convert to PyTorch sparse tensor
    coo = normalized_adj.tocoo()
    indices = torch.tensor(np.vstack((coo.row, coo.col)), dtype=torch.long)
    values = torch.tensor(coo.data, dtype=torch.float32)
    shape = torch.Size(coo.shape)
    
    return torch.sparse_coo_tensor(indices, values, shape).coalesce()

In [10]:
class StandardScaler:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        return (data * self.std) + self.mean
    
class TrafficDataset(Dataset):
    def __init__(self, data, seq_len=12, horizon=12):
        """
        data: shape (Total_Time_Steps, Nodes, Features)
        Features = 2 (0: Speed, 1: Time of Day)
        """
        self.data = data
        self.seq_len = seq_len
        self.horizon = horizon

    def __len__(self):
        return len(self.data) - self.seq_len - self.horizon + 1

    def __getitem__(self, index):
        # Input X: [seq_len, nodes, 2] -> Contém Velocidade e Horário
        X = self.data[index : index + self.seq_len]
        
        # Target Y: [horizon, nodes, 1] -> Contém APENAS a Velocidade (feature 0)
        # Cortamos a feature de horário na saída porque não precisamos prevê-la
        Y = self.data[index + self.seq_len : index + self.seq_len + self.horizon, :, 0:1]
        
        # PyTorch espera o formato [Features, Nodes, Time] para a entrada da convolução
        X = torch.tensor(X, dtype=torch.float32).permute(2, 1, 0)
        Y = torch.tensor(Y, dtype=torch.float32)
        
        return X, Y
    

def get_dataloaders(h5_file, batch_size=64, seq_len=12, horizon=12):
    # 1. Carrega os dados brutos ignorando as strings legadas
    with h5py.File(h5_file, 'r') as f:
        # Extrai a matriz de velocidades (34272, 207)
        speed_matrix = f['df']['block0_values'][:]
        # Extrai os timestamps (em nanossegundos desde 1970)
        timestamp_ns = f['df']['axis1'][:]
        
    # Shape resultante da velocidade: (Tempo, Nós, 1)
    speed_data = np.expand_dims(speed_matrix, axis=-1)
    
    # 2. Converte os nanossegundos para Datetime e calcula a "Hora do Dia" (ToD)
    timestamps = pd.to_datetime(timestamp_ns)
    tod = (timestamps.hour * 60 + timestamps.minute) / (24 * 60)
    
    # 3. Formata o ToD para bater com as dimensões espaciais (Tempo, Nós, 1)
    # np.tile replica o horário atual para os 207 sensores
    tod_data = np.tile(tod.values.reshape(-1, 1, 1), (1, speed_matrix.shape[1], 1))
    
    # 4. Concatena Velocidade e ToD
    # Shape final dos dados: (Tempo, Nós, 2)
    data = np.concatenate([speed_data, tod_data], axis=-1)
    
    # 5. Divisão Padrão: 70% Treino, 10% Validação, 20% Teste
    num_samples = data.shape[0]
    num_train = round(num_samples * 0.7)
    num_val = round(num_samples * 0.1)
    
    train_data = data[:num_train].copy()
    val_data = data[num_train:num_train+num_val].copy()
    test_data = data[num_train+num_val:].copy()
    
    # 6. Fit do Scaler APENAS no Treino e APENAS na Velocidade (Feature 0)
    mean = train_data[..., 0].mean()
    std = train_data[..., 0].std()
    scaler = StandardScaler(mean, std)
    
    # Aplica o scaling na Feature 0
    train_data[..., 0] = scaler.transform(train_data[..., 0])
    val_data[..., 0]   = scaler.transform(val_data[..., 0])
    test_data[..., 0]  = scaler.transform(test_data[..., 0])
    
    # Cria os PyTorch Datasets (garanta que o seu TrafficDataset já esteja 
    # atualizado para lidar com 2 features na entrada e 1 na saída)
    train_dataset = TrafficDataset(train_data, seq_len, horizon)
    val_dataset   = TrafficDataset(val_data, seq_len, horizon)
    test_dataset  = TrafficDataset(test_data, seq_len, horizon)
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, scaler



In [11]:
adj_mx = load_adj_matrix('data/adj_METR-LA.pkl')
print(f"Adjacency Matrix Shape: {adj_mx.shape}")

train_loader, val_loader, test_loader, scaler = get_dataloaders(h5_file='data/metr-la.h5', batch_size=64)

for X, Y in train_loader:
    print(f"X shape: {X.shape}") # Expected: [64, 1, 207, 12]
    print(f"Y shape: {Y.shape}") # Expected: [64, 12, 207, 1]
    break

print(len(train_loader))
print(len(val_loader))
print(len(test_loader))

Adjacency Matrix Shape: torch.Size([207, 207])
X shape: torch.Size([64, 2, 207, 12])
Y shape: torch.Size([64, 12, 207, 1])
374
54
107
